# Whale / Boat / Water Chip Classifier — Training Notebook

Trains three models to give each 500x500 SkySat chip one label: **water**, **whale**, or **boat**.

| Model | Taken from |
| --- | --- |
| EfficientNet-B0 (CNN) | ICS 605 AI-image detector |
| DeiT-Tiny (ViT) | same notebook |
| SatViT-V2 encoder + classification head | Joe's flood project, `satvit/` |

The training loop, early stopping and checkpointing are the ones from the 605 notebook. The SatViT
encoder, weight loading, partial freezing and per-band normalization are Joe's. Data loading,
splits and model inputs live in `src/whaleboat/` so `evaluate_models.ipynb` processes chips
exactly the same way.

**Data** (pick with `DATASETS`):

- `original`: the 144 hand-picked chips in `data/whale/data/` (`notebooks/data_ingester.ipynb`).
- `skysat`: `data/skysat_chips_500/main/`, chips cut from the annotated SkySat scenes by
  `scripts/build_skysat_chips.py` and cleaned by `scripts/qc_skysat_chips.py` (water, boat, and
  whales with certainty 1-2).

**SatViT weights:** download `SatViT_V2.pt` (link in `satvit/README.md`) into `checkpoints/`.
Without it the SatViT run is skipped and the other two still train.

**Outputs:** everything for one run goes to `checkpoints/<RUN_NAME>/`: `run.json` (datasets, the
exact train/val/test chip lists, band stretch, seed, hyperparameters) and one `<model>.pt` per
model (best weights + learning history). This notebook only reports **validation** scores; the
test set is scored once, in `notebooks/evaluate_models.ipynb`.

## 1. Setup

In [ ]:
import os
import sys

# Has to be set before torch starts CUDA. With the default module loading, torch's CUDA 12.6
# runtime fails on a 12.2 driver with "CUDA-capable device(s) is/are busy or unavailable".
os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')

if 'google.colab' in sys.modules:
    !pip install -q rasterio einops timm

In [ ]:
import copy
import gc
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()),
    Path.cwd(),
)
sys.path.insert(0, str(ROOT / 'src'))
from whaleboat.data import (  # noqa: E402
    CLASSES,
    IMAGENET_MEAN,
    IMAGENET_STD,
    band_stretch,
    describe_split,
    load_chips,
    make_loader,
    make_split,
)
from whaleboat.models import MODELS, build_model  # noqa: E402

print(f'Python:  {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no CUDA detected. Training will be very slow on CPU.')

## 2. Config

`RUN_NAME` names the output folder, so ablations don't overwrite each other. For an ablation,
set `SPLIT_FROM` to the main run's name: this run then reuses that run's split (keeping only the
chips in its own `DATASETS`), so both are scored on the same test chips.

In [ ]:
ORIG_DIR = ROOT / 'data' / 'whale' / 'data'
SKYSAT_DIR = ROOT / 'data' / 'skysat_chips_500'
SATVIT_DIR = ROOT / 'satvit'
CKPT_DIR = ROOT / 'checkpoints'
SATVIT_WEIGHTS = CKPT_DIR / 'SatViT_V2.pt'

RUN_NAME = 'main'
# Chip sets to train on: 'original' (144 chips), 'skysat' (skysat_chips_500/main), or both.
DATASETS = ['original', 'skysat']
# 'group': overlapping chips of one day stay in one split (new places, seen conditions).
# 'date': whole days stay in one split (new days / weather; stricter, lumpier).
# 'random': stratified random split; overlapping chips leak across splits. Comparison only.
SPLIT_MODE = 'group'
SPLIT_FROM = None  # e.g. 'main' to reuse that run's split for an ablation
MODELS_TO_TRAIN = ['efficientnet_b0', 'deit_tiny', 'satvit']

NUM_EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
NUM_WORKERS = 2
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
RUN_DIR = CKPT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')
print(f'Run:    {RUN_DIR}  data={DATASETS}  split={SPLIT_FROM or SPLIT_MODE}')

## 3. Load chip list and labels

Only paths and labels are loaded here; each chip is read from disk when a model needs it.

- **original:** the label comes from which mask folder has a file with the chip's name; chips in
  neither are water. There are no dates, but the chips overlap heavily, so chips that overlap
  (directly or through a chain) share a location group.
- **skysat:** the label comes from `manifest.csv`. Planet images an area several times seconds
  apart, so chips from one day whose footprints overlap (directly or through a chain) share a
  group: they can show the very same boat.

A chip with both a whale and a boat counts as whale.

In [ ]:
chips = load_chips(DATASETS, ORIG_DIR, SKYSAT_DIR)
by_name = {c['name']: c for c in chips}

print(f'{len(chips)} chips in {len({c["group"] for c in chips})} groups')
for s in sorted({c['source'] for c in chips}):
    cs = [c for c in chips if c['source'] == s]
    counts = '  '.join(f'{k}={sum(c["label"] == k for c in cs)}' for k in CLASSES)
    print(f'  {s:9s} {len(cs):5d}   {counts}')

## 4. Train / val / test split

With `SPLIT_MODE = 'group'`, whole groups go to one split, chosen by `StratifiedGroupKFold` so
class ratios stay roughly even. Overlapping chips in train and test would otherwise let a model
score well by recognising the place or the very same boat. Test chips then come from the same
days as training chips (different places), so this measures new places under familiar weather.
`SPLIT_MODE = 'date'` holds out whole days instead, which tests new weather but leaves only a
few days per split. Running both (with different `RUN_NAME`s) is a useful comparison.

The original chips form only 4 large groups, so with `DATASETS = ['original']` alone a split can
miss a class (the counts show it).

The split and the band stretch (fitted on training chips only) are saved to `run.json`.

In [ ]:
if SPLIT_FROM:
    base = json.loads((CKPT_DIR / SPLIT_FROM / 'run.json').read_text())['split']
    split = {part: [n for n in names if n in by_name] for part, names in base.items()}
else:
    split = make_split(chips, SPLIT_MODE, VAL_SPLIT, TEST_SPLIT, SEED)
describe_split(chips, split)

train_chips = [by_name[n] for n in split['train']]
val_chips = [by_name[n] for n in split['val']]
BAND_LO, BAND_HI = band_stretch([c['path'] for c in train_chips], seed=SEED)

run_info = {
    'run_name': RUN_NAME,
    'created': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'datasets': DATASETS,
    'split_mode': SPLIT_MODE,
    'split_from': SPLIT_FROM,
    'seed': SEED,
    'classes': CLASSES,
    'band_lo': BAND_LO,
    'band_hi': BAND_HI,
    'hyperparameters': {
        'epochs': NUM_EPOCHS,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'patience': PATIENCE,
        'val_split': VAL_SPLIT,
        'test_split': TEST_SPLIT,
        'models': MODELS,
    },
    'split': split,
}
(RUN_DIR / 'run.json').write_text(json.dumps(run_info, indent=1))
print(f'saved {RUN_DIR / "run.json"}')

## 5. Model inputs

Each model family gets the input it was pretrained on (`whaleboat.data.to_model_input`):

- **EfficientNet / DeiT:** RGB only. Each band gets one fixed 2nd–98th percentile stretch, taken
  from the training chips, then the ImageNet normalization the 605 notebook used.
- **SatViT:** Joe's per-chip, per-band z-score. SatViT-V2 expects 15 bands (2 Sentinel-1, then
  Sentinel-2 B1–B12 with B8A). Our Blue, Green, Red and NIR go where Sentinel-2 B2, B3, B4 and B8
  sit, and the other 11 bands are left at 0.

Chips are resized from 500 to the model's input size. Training adds random flips (as in Joe's
loop) and 90° rotations, since overhead imagery has no "up".

In [ ]:
def loaders_for(name):
    cfg = MODELS[name]
    kw = {
        'img_size': cfg['img_size'],
        'satvit': name == 'satvit',
        'band_lo': BAND_LO,
        'band_hi': BAND_HI,
        'batch_size': cfg['batch_size'],
        'workers': NUM_WORKERS,
    }
    return make_loader(train_chips, train=True, **kw), make_loader(val_chips, **kw)


# Sanity check: one training batch as the CNN/ViT sees it.
imgs, batch_labels, batch_names = next(iter(loaders_for('efficientnet_b0')[0]))
print(imgs.shape, batch_labels.tolist())
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, lab, name in zip(axes, imgs, batch_labels, batch_names):
    ax.imshow((img * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0))
    ax.set_title(f'{CLASSES[lab]} ({name.split("/")[-1][:24]})', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Models

`whaleboat.models.build_model`: EfficientNet-B0 and DeiT-Tiny come from `timm` exactly as in the
605 notebook, with 3 output classes instead of 2. For SatViT, the encoder, weight loading and
freezing (only encoder layers 10 and 11 train) are Joe's code from `satvit/`, unchanged. His
flood decoder upsampled all 1024 patch encodings back into a mask; for one label per chip, the
head here averages the patch encodings and maps them to class scores.

## 7. Train loop

`run_epoch` and `train_model` are the 605 versions with these changes:

- The loss is cross-entropy weighted by class (inverse training frequency), so the most common
  class in the training split doesn't swamp the others.
- The optimizer only gets parameters that are set to train (SatViT's frozen layers).
- Each epoch also records validation macro F1. The best epoch is still picked by validation loss.
- The checkpoint stores the learning history for the evaluation notebook's learning curves.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for imgs, labels_, _ in loader:
        imgs, labels_ = imgs.to(DEVICE, non_blocking=True), labels_.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train_mode):
            logits = model(imgs)
            loss = criterion(logits, labels_)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels_.cpu().tolist())

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / n, accuracy_score(all_labels, all_preds), macro_f1


def train_model(name):
    print(f'\n{"=" * 60}\nTRAINING: {name}\n{"=" * 60}')
    train_loader, val_loader = loaders_for(name)
    model = build_model(name, len(CLASSES), SATVIT_DIR, SATVIT_WEIGHTS).to(DEVICE)

    counts = np.bincount([CLASSES.index(c['label']) for c in train_chips], minlength=len(CLASSES))
    class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best = {'val_loss': float('inf')}
    epochs_no_improve = 0
    history = {k: [] for k in ['train_loss', 'val_loss', 'train_f1', 'val_f1']}
    ckpt_path = RUN_DIR / f'{name}.pt'

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, vl_f1 = run_epoch(model, val_loader, criterion)
        scheduler.step()
        for k, v in zip(history, [tr_loss, vl_loss, tr_f1, vl_f1]):
            history[k].append(v)
        print(
            f'[{name}] epoch {epoch:02d}/{NUM_EPOCHS}  '
            f'train_loss={tr_loss:.4f} train_f1={tr_f1:.3f}  '
            f'val_loss={vl_loss:.4f} val_f1={vl_f1:.3f} val_acc={vl_acc:.3f}  '
            f'({time.time() - t0:.0f}s)'
        )
        if vl_loss < best['val_loss']:
            best = {
                'model_state_dict': copy.deepcopy(model.state_dict()),
                'epoch': epoch,
                'val_loss': vl_loss,
                'val_acc': vl_acc,
                'val_macro_f1': vl_f1,
            }
            epochs_no_improve = 0
            print('    ↳ new best')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'    ↳ early stopping at epoch {epoch} (best was {best["epoch"]}).')
                break

    torch.save(
        {
            **best,
            'model_name': name,
            'img_size': MODELS[name]['img_size'],
            'classes': CLASSES,
            'class_weights': class_weights.tolist(),
            'history': history,
            'run_name': RUN_NAME,
        },
        ckpt_path,
    )
    print(f'saved best epoch {best["epoch"]} → {ckpt_path}')
    return best, history


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (a, b, ylabel) in zip(
        axes,
        [('train_loss', 'val_loss', 'weighted cross-entropy'), ('train_f1', 'val_f1', 'macro F1')],
    ):
        ax.plot(history[a], label='train')
        ax.plot(history[b], label='validation')
        ax.set_xlabel('epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
    fig.suptitle(title)
    plt.show()

## 8. Train each model

In [ ]:
results = {}


def run(name):
    if name == 'satvit' and not SATVIT_WEIGHTS.exists():
        print(f'Skipping SatViT: put SatViT_V2.pt at {SATVIT_WEIGHTS} (link in satvit/README.md).')
        return
    best, history = train_model(name)
    plot_history(history, f'{name} learning curves ({RUN_NAME})')
    results[name] = {k: best[k] for k in ['epoch', 'val_loss', 'val_acc', 'val_macro_f1']}
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
run('efficientnet_b0')

In [ ]:
run('deit_tiny')

In [ ]:
run('satvit')

## 9. Validation summary

Validation scores of each model's best epoch (n = validation chips). These guided training
(early stopping); they are not the final result. Test scores, baselines, curves and error
analysis are in `notebooks/evaluate_models.ipynb`.

In [ ]:
print(f'run={RUN_NAME}  validation n={len(val_chips)}')
print(f'{"model":18s} {"epoch":>5s} {"val loss":>9s} {"val acc":>8s} {"val macro F1":>13s}')
for name, r in results.items():
    print(
        f'{name:18s} {r["epoch"]:5d} {r["val_loss"]:9.4f} {r["val_acc"]:8.3f} '
        f'{r["val_macro_f1"]:13.3f}'
    )